In [1]:
from IPython.display import HTML
HTML('''
<script
    src="https://cdnjs.cloudflare.com/ajax/libs/jquery/2.0.3/jquery.min.js ">
</script>
<script>
code_show=true; 
function code_toggle() {
 if (code_show){
 $('div.jp-CodeCell > div.jp-Cell-inputWrapper').hide();
 } else {
$('div.jp-CodeCell > div.jp-Cell-inputWrapper').show();
 }
 code_show = !code_show
} 
$( document ).ready(code_toggle);
</script>
<form action="javascript:code_toggle()"><input type="submit"
    value="Click here to toggle on/off the raw code."></form>
'''
)

<div align="center">
    <img src="figures/header.png" alt="title_header">
</div>

<h2 style=" color: #212618; font-weight:600;">Introduction</h2>

Growing regulatory pressure and stakeholder scrutiny have made ESG (Environmental, Social, and Governance) reporting a central concern for supply chain management. This is particularly a case in developing markets where data quality and reporting maturity vary widely. A persistent challenge in this domain is **greenwashing**: the systematic over-reporting of positive indicators and under-reporting of harmful ones by suppliers who lack the governance infrastructure to ensure honest disclosure.

We introduce **ProcGen**, a synthetic supply chain ESG data generator designed around the Philippine business context. ProcGen simulates realistic ESG reporting behavior by embedding latent variables—supplier maturity, efficiency, grid intensity, and regulatory pressure—that jointly drive both the **true underlying performance** and the **reported values**, reproducing the reporting bias patterns seen in real-world ESG surveys.

<h2 style=" color: #212618; font-weight:600;">Methodology</h2>

<h3 style=" color: #212618; font-weight:600;">Overview</h3>

ProcGen generates data across six relational tables linked by `supplier_id`. The generation follows a strict dependency order to enforce causal consistency:

1. **`indicator_definitions`** — defines the 8 ESG indicators (E/S/G), their metric types, bias directions, and survey methods.
2. **`suppliers`** — the entity backbone; each supplier carries latent variables (`esg_maturity`, `efficiency_factor`) and contextual attributes (`region`, `industry`, `grid_intensity`, `regulatory_pressure`).
3. **`esg_surveys`** — the core bias model; for each supplier × indicator, a `true_value` is generated mechanistically and a `reported_value` is produced by applying a maturity-driven bias: `bias_mag = α × maturity^β`.
4. **`audits`** — sparse, high-trust third-party verifications generated probabilistically (larger, more regulated suppliers are audited more frequently).
5. **`emissions_estimates`** — modeled Scope 3 approximations using spend-based methods.
6. **`esg_scores`** — derived output scoring each supplier 0–100 per ESG pillar, aggregated as `0.4E + 0.3S + 0.3G`, and bucketed into risk tiers.

<h3 style=" color: #212618; font-weight:600;">Model Assumptions</h3>

1. **Latent maturity drives both performance and bias simultaneously.** A higher `esg_maturity` score produces genuinely better true values *and* more aggressive favorable reporting—capturing the paradox where sophisticated reporters are also better at greenwashing.
2. **Regulatory pressure attenuates bias.** High-pressure regions apply a `reg_factor = 0.6` multiplier to the bias term, modelling the effect of enforcement on reporting honesty.
3. **Survey method degrades data reliability.** `direct` reporting retains full bias potential; `indirect` and `estimated` methods apply reduction factors of 0.6 and 0.3 respectively.
4. **Binary governance indicators use flip probability** rather than continuous bias, with misreport probability governed by `method_factor` and regulatory pressure.
5. **Audits are sparse by design** (~10% base rate annually), reflecting real procurement audit coverage in Philippine supply chains.

<div style="font-size:14px; font-style:default; text-align:center">
    <b>Table 1. Database Schema</b><br><br>

| Table | Grain | Key Columns |
| :--- | :--- | :--- |
| **`suppliers`** | 1 row per supplier | `id`, `industry`, `region`, `size_tier`, `esg_maturity`, `efficiency_factor`, `grid_intensity`, `regulatory_pressure` |
| **`indicator_definitions`** | 1 row per indicator | `id`, `category`, `subcategory`, `metric_type`, `bias_direction`, `survey_method` |
| **`esg_surveys`** | supplier × indicator × period | `supplier_id`, `indicator_id`, `true_value`, `reported_value`, `data_quality_flag` |
| **`audits`** | supplier × audit event | `supplier_id`, `audit_type`, `audit_score`, `findings_count` |
| **`esg_scores`** | supplier × period | `supplier_id`, `environment_score`, `social_score`, `governance_score`, `overall_esg_score`, `risk_tier` |

</div>

<h2 style=" color: #212618; font-weight:600;">Use Cases</h2>

To showcase the data, we walk through four use cases below that span exploratory profiling, greenwashing detection, risk segmentation, and audit analytics. All analyses load directly from `esg_simulation.db`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine

DB_PATH = "data/esg_simulation.db"  # adjust path as needed
engine = create_engine(f"sqlite:///{DB_PATH}", future=True)

suppliers = pd.read_sql("SELECT * FROM suppliers", engine)
indicators = pd.read_sql("SELECT * FROM indicator_definitions", engine)
surveys = pd.read_sql("SELECT * FROM esg_surveys", engine)
audits = pd.read_sql("SELECT * FROM audits", engine)
scores = pd.read_sql("SELECT * FROM esg_scores", engine)

# Merge dataframes
df_scores = scores.merge(suppliers, left_on="supplier_id", right_on="id", suffixes=("", "_sup"))

df_surveys = (
    surveys
    .merge(indicators, left_on="indicator_id", right_on="id", suffixes=("", "_ind"))
    .merge(suppliers,  left_on="supplier_id",  right_on="id",  suffixes=("", "_sup"))
)
df_surveys["bias_pct"] = (
    (df_surveys["reported_value"] - df_surveys["true_value"]) / df_surveys["true_value"].abs() * 100
)

print(f"Suppliers: {len(suppliers)}  |  Survey records: {len(surveys)}  |  Audits: {len(audits)}")

Suppliers: 100  |  Survey records: 800  |  Audits: 13


<h3 style=" color: #212618; font-weight:600;">1. Supplier ESG Portfolio Overview</h3>

The first use case is a portfolio-level snapshot of all simulated Philippine suppliers. This helps the procurement analyst assess the current landscape based on ESG pillars, segmenting locations and industry that pose the most risk.

In [ ]:
n_total    = len(df_scores)
n_high     = (df_scores["risk_tier"] == "high").sum()
n_med      = (df_scores["risk_tier"] == "med").sum()
n_low      = (df_scores["risk_tier"] == "low").sum()
avg_score  = df_scores["overall_esg_score"].mean()
avg_env    = df_scores["environment_score"].mean()
avg_soc    = df_scores["social_score"].mean()
avg_gov    = df_scores["governance_score"].mean()

print(f"Portfolio: {n_total} suppliers  |  Avg ESG: {avg_score:.1f}")
print(f"Risk tiers → High: {n_high}  Med: {n_med}  Low: {n_low}")
print(f"Pillar averages → E: {avg_env:.1f}  S: {avg_soc:.1f}  G: {avg_gov:.1f}")

Portfolio: 100 suppliers  |  Avg ESG: 75.8
Risk tiers → High: 0  Med: 30  Low: 70
Pillar averages → E: 95.3  S: 80.3  G: 45.5


In [ ]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("ESG Score Distribution", "Risk Tier by Industry", "ESG Pillar Averages by Region"),
    column_widths=[0.30, 0.35, 0.35]
)

# Histogram of ESG Scores
color_map = {"low": "#10b981", "med": "#f59e0b", "high": "#ef4444"}
for tier, grp in df_scores.groupby("risk_tier"):
    fig.add_trace(
        go.Histogram(x=grp["overall_esg_score"], name=tier.capitalize(),
                     marker_color=color_map[tier], opacity=0.8, nbinsx=15,
                     legendgroup=tier),
        row=1, col=1
    )

# Risk Distribution by Industry
rt_ind = df_scores.groupby(["industry", "risk_tier"]).size().reset_index(name="count")
for tier in ["high", "med", "low"]:
    sub = rt_ind[rt_ind["risk_tier"] == tier]
    fig.add_trace(
        go.Bar(x=sub["industry"], y=sub["count"], name=tier.capitalize(),
               marker_color=color_map[tier], legendgroup=tier, showlegend=False),
        row=1, col=2
    )

# ESG Breakdown per Region
region_pillar = df_scores.groupby("region")[["environment_score","social_score","governance_score"]].mean().reset_index()
for col_name, color in zip(["environment_score","social_score","governance_score"],
                            ["#34d399", "#60a5fa", "#a78bfa"]):
    fig.add_trace(
        go.Bar(x=region_pillar["region"], y=region_pillar[col_name],
               name=col_name.replace("_score","").capitalize(),
               marker_color=color, showlegend=False),
        row=1, col=3
    )

fig.update_layout(
    barmode="stack", height=400,
    title_text="<b>ESG Portfolio Overview</b>",
    template="plotly_white", legend_title="Risk Tier"
)
fig.update_xaxes(tickangle=20)
fig.show()

<div style="font-size:14px; font-style:default; text-align:center">
    <b>Figure 1. ESG Portfolio Overview</b><br><br>
</div>

The portfolio skews toward medium-to-high overall scores, with no suppliers falling into the high-risk tier under this simulation. Generally this reflects that the generator's default parameters produce a reasonably mature baseline set of suppliers. Looking at the ESG pillars, governance appears weak in scores because it is harder to achieve for small and medium suppliers. Retail dominates the supplier mix, and the regional spread across Visayas, Mindanao, and Luzon reflects the Philippine supply base distribution embedded in the generator.

<h3 style=" color: #212618; font-weight:600;">2. Greenwashing Detection</h3>

The core scientific contribution of ProcGen is its explicit separation of `true_value` and `reported_value` in the `esg_surveys` table. This allows us to directly measure **reporting bias** from the difference of values and examine how it correlates with maturity, industry, and regulatory environment. This is the core engine to supplement greenwashing detection by simulating it.

In [5]:
# Filter to continuous/percent indicators only (binary bias_pct is undefined)
df_cont = df_surveys[df_surveys["metric_type"].isin(["continuous", "percent"])].copy()

# ── Bias by indicator ────────────────────────────────────────────────────────
bias_ind = (
    df_cont.groupby(["indicator_id", "category", "bias_direction"])["bias_pct"]
    .agg(["mean", "std"])
    .reset_index()
    .rename(columns={"mean": "avg_bias_pct", "std": "std_bias_pct"})
)
bias_ind["expected_direction"] = bias_ind["bias_direction"].map({1: "Inflate (+)", -1: "Deflate (-)"})

fig_bias = px.bar(
    bias_ind.sort_values("avg_bias_pct"),
    x="avg_bias_pct", y="indicator_id", color="category",
    error_x="std_bias_pct",
    color_discrete_map={"E": "#34d399", "S": "#60a5fa", "G": "#a78bfa"},
    orientation="h",
    labels={"avg_bias_pct": "Avg Reporting Bias (%)", "indicator_id": "Indicator"},
    title="<b>Average Reporting Bias per ESG Indicator</b><br><sup>Positive = over-reported; Negative = under-reported (relative to true value)</sup>",
    template="plotly_white", height=380
)
fig_bias.add_vline(x=0, line_dash="dash", line_color="#6b7280")
fig_bias.show()

<div style="font-size:14px; font-style:default; text-align:center">
    <b>Figure 2.a Greenwashing Bias Analysis</b><br><br>
</div>

In [ ]:
# Bias magnitude vs. ESG maturity scatter
df_cont["abs_bias"] = df_cont["bias_pct"].abs()
maturity_bias = df_cont.groupby("supplier_id").agg(
    esg_maturity=("esg_maturity", "first"),
    avg_abs_bias=("abs_bias", "mean"),
    industry=("industry", "first"),
    size_tier=("size_tier", "first"),
    regulatory_pressure=("regulatory_pressure", "first")
).reset_index()

fig_mat = px.scatter(
    maturity_bias, x="esg_maturity", y="avg_abs_bias",
    color="regulatory_pressure", symbol="size_tier",
    color_discrete_map={"low": "#f87171", "med": "#fbbf24", "high": "#34d399"},
    hover_data=["industry", "supplier_id"],
    trendline="ols",
    labels={"esg_maturity": "Latent ESG Maturity", "avg_abs_bias": "Avg Absolute Reporting Bias (%)"},
    title="<b>ESG Maturity vs. Reporting Bias Magnitude</b><br><sup>Higher maturity → larger absolute bias; regulatory pressure attenuates it</sup>",
    template="plotly_white", height=420
)
fig_mat.show()

<div style="font-size:14px; font-style:default; text-align:center">
    <b>Figure 2.b Greenwashing Bias Analysis</b><br><br>
</div>

The bias charts above should be read as a **validation of the simulation model**, not as a
real-world detection output. Greenwashing is difficult to detect because ground truth is unobservable in nature.

What ProcGen enables is using this separation as a **benchmarking environment**:
detection algorithms (outlier models, peer-group comparisons, audit cross-referencing) can be
developed and evaluated against the reported side of the data, then validated against the true
side as a known label.

At first glance of Figure 2.a, you can observe that factors in scoring tended to be overreported or underreported based on its implication. It is good to have lower emissions or have higher training hours, evidence of reporting bias of suppliers. Furthermore, the charts in Figure 2.b show that suppliers report better numbers than reality, and those with higher maturity or looser regulatory oversight do it more. For real detection, analysts rely on indirect signals like third-party audits and modeled emissions estimates, both of which Use Case 4 explores.

<h3 style=" color: #212618; font-weight:600;">3. ESG Risk Segmentation</h3>

A practical procurement application is risk-tiering suppliers for due diligence prioritization. This section compares suppliers across multiple dimensions simultaneously, and introduces the gap between a score computed from **reported** survey values versus one computed from **true** values — quantifying how much greenwashing inflates a supplier's apparent ESG standing.

In [ ]:

def percentile_score(series, higher_is_better=True):
    ranks = series.rank(pct=True) * 100
    return ranks if higher_is_better else 100 - ranks

cont_ind = df_surveys[df_surveys["metric_type"].isin(["continuous", "percent"])].copy()

# score each indicator on true vs reported
for val_col, score_col in [("true_value", "true_score"), ("reported_value", "rep_score")]:
    grps = []
    for ind_id, grp in cont_ind.groupby("indicator_id"):
        hib = grp["bias_direction"].iloc[0] == 1
        tmp = grp[["supplier_id", val_col]].copy()
        tmp[score_col] = percentile_score(grp[val_col], higher_is_better=hib).values
        grps.append(tmp)
    merged = pd.concat(grps).groupby("supplier_id")[score_col].mean()
    df_scores[score_col] = df_scores["supplier_id"].map(merged)

df_scores["score_inflation"] = df_scores["rep_score"] - df_scores["true_score"]

# ── Bubble chart: maturity × true score × score inflation ───────────────────
fig_seg = px.scatter(
    df_scores.dropna(subset=["true_score","rep_score"]),
    x="esg_maturity", y="true_score",
    size=df_scores["score_inflation"].abs().clip(lower=0.5),
    color="score_inflation",
    color_continuous_scale="RdYlGn",
    color_continuous_midpoint=0,
    hover_data=["industry", "region", "size_tier", "risk_tier", "supplier_id"],
    facet_col="size_tier",
    labels={"esg_maturity": "ESG Maturity", "true_score": "True ESG Score (proxy)",
            "score_inflation": "Score Inflation"},
    title="<b>Supplier Segmentation: True ESG Performance vs. Reported Score Inflation</b><br>"
          "<sup>Bubble size = magnitude of inflation | Green = over-reported | Red = under-reported</sup>",
    template="plotly_white", height=440
)
fig_seg.show()

In [8]:
# ── Box: overall_esg_score by industry, split by size ───────────────────────
fig_box = px.box(
    df_scores, x="industry", y="overall_esg_score",
    color="size_tier",
    color_discrete_map={"Small": "#93c5fd", "Medium": "#6366f1",
                        "Large": "#312e81", "Enterprise": "#0f172a"},
    points="all", notched=False,
    labels={"overall_esg_score": "Overall ESG Score", "industry": "Industry"},
    title="<b>ESG Score Distribution by Industry and Size Tier</b>",
    template="plotly_white", height=400
)
fig_box.show()

<div style="font-size:14px; font-style:default; text-align:center">
    <b>Figure 3. ESG Risk Segmentation</b><br><br>
</div>

The bubble chart shows the segmentation most useful for procurement risk teams. Suppliers in the upper-left quadrant (high true score, low maturity) are genuinely performing well despite limited ESG infrastructure. Suppliers in the lower-right (low true score, high maturity) have the capability to articulate favorable ESG narratives while underlying performance remains weak, representing the highest greenwashing risk. The score inflation coloring confirms this reading. The industry box plots reveal that Manufacturing suppliers show the widest variance, a reflection of their broader range of grid intensity and operational scale in the Philippine context.

<h3 style=" color: #212618; font-weight:600;">4. Audit Analytics — Where Independent Verification Makes a Difference</h3>

Audits occur rarely in the simulation but carry significantly higher weight in the scoring model. This section profiles audit outcomes, examines whether audited suppliers' corrected scores diverge meaningfully from their self-reported baselines, and surfaces which supplier characteristics predict audit findings severity.

In [ ]:
df_audits_full = audits.merge(suppliers, left_on="supplier_id", right_on="id")
df_audits_full = df_audits_full.merge(
    scores[["supplier_id","overall_esg_score","risk_tier"]],
    on="supplier_id"
)

fig_aud = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Audit Score vs. Self-Reported ESG Score",
                    "Findings Count by Audit Type & Industry")
)

# Audit Score vs Self-reported ESG
fig_aud.add_trace(
    go.Scatter(
        x=df_audits_full["overall_esg_score"],
        y=df_audits_full["audit_score"],
        mode="markers+text",
        text=df_audits_full["supplier_id"].astype(str),
        textposition="top center",
        marker=dict(
            size=df_audits_full["findings_count"] * 5 + 8,
            color=df_audits_full["findings_count"],
            colorscale="OrRd",
            showscale=True,
            colorbar=dict(title="Findings", x=0.45)
        ),
        name="Audited Suppliers"
    ),
    row=1, col=1
)
# 45-degree reference line
lim = [30, 100]
fig_aud.add_trace(
    go.Scatter(x=lim, y=lim, mode="lines",
               line=dict(dash="dash", color="#9ca3af"),
               name="Parity line", showlegend=True),
    row=1, col=1
)

# Findings count by Audit Type and Industry
agg = df_audits_full.groupby(["industry","audit_type"])["findings_count"].mean().reset_index()
for atype, color in zip(["site","document","third_party"], ["#ef4444","#f59e0b","#6366f1"]):
    sub = agg[agg["audit_type"] == atype]
    if not sub.empty:
        fig_aud.add_trace(
            go.Bar(x=sub["industry"], y=sub["findings_count"],
                   name=atype.replace("_"," ").title(),
                   marker_color=color),
            row=1, col=2
        )

fig_aud.update_xaxes(title_text="Self-Reported ESG Score", row=1, col=1)
fig_aud.update_yaxes(title_text="Audit Score", row=1, col=1)
fig_aud.update_xaxes(title_text="Industry", row=1, col=2)
fig_aud.update_yaxes(title_text="Avg Findings Count", row=1, col=2)
fig_aud.update_layout(
    barmode="group", height=430, template="plotly_white",
    title_text="<b>Audit Analytics — Independent Verification Outcomes</b>",
    legend=dict(orientation="h", yanchor="bottom", y=-0.25)
)
fig_aud.show()

In [ ]:
# Score adjustments after Audit
df_audits_full["blended_score"] = (
    0.7 * df_audits_full["overall_esg_score"] + 0.3 * df_audits_full["audit_score"]
)
df_audits_full["score_delta"] = df_audits_full["blended_score"] - df_audits_full["overall_esg_score"]

fig_delta = go.Figure()
fig_delta.add_trace(go.Bar(
    x=df_audits_full["supplier_id"].astype(str),
    y=df_audits_full["score_delta"],
    marker_color=["#ef4444" if d < 0 else "#10b981" for d in df_audits_full["score_delta"]],
    text=df_audits_full["score_delta"].round(1),
    textposition="outside",
    hovertemplate=(
        "Supplier %{x}<br>Score Δ: %{y:.1f}<br>"
        "<extra></extra>"
    )
))
fig_delta.add_hline(y=0, line_dash="solid", line_color="#374151")
fig_delta.update_layout(
    title="<b>Audit Adjustment Effect — Score Δ after Blending Audit Results</b><br>"
          "<sup>Red = audit lowers the score (audit found worse performance); Green = audit raises it</sup>",
    xaxis_title="Supplier ID", yaxis_title="Score Change (Blended − Reported)",
    template="plotly_white", height=360
)
fig_delta.show()

<div style="font-size:14px; font-style:default; text-align:center">
    <b>Figure 4. Audit Analytics</b><br><br>
</div>

To put Figure 4 simply, suppliers above the diagonal line scored similarly in both their own reporting and the independent audit with honest reporting. Those below it reported better than auditors
found, with larger bubbles flagging the ones with the most issues uncovered. The bar chart
below shows how much each supplier's score shifts once audit results are factored in — a
drop signals a supplier worth scrutinizing further. With only 13 out of 100 suppliers
audited, the charts also make the case for expanding audit coverage.